In [1]:
import csv
from pathlib import Path

In [2]:
actual_path = Path("raw-zip-actual")
actual_files = sorted(actual_path.glob("*.zip"))
print(len(actual_files))
actual_files[:5]

230


[PosixPath('raw-zip-actual/20060101RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060201RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060301RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060401RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060501RTLineOutages_csv.zip')]

In [3]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"))
print(len(scheduled_files))
scheduled_files[:5]

218


[PosixPath('raw-zip-scheduled/20080101SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080301SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080401SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20080501SCLineOutages_csv.zip')]

In [4]:
import re
from collections import defaultdict, namedtuple

ActualOutage = namedtuple(
    "ActualOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "outage_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)
ScheduledOutage = namedtuple(
    "ScheduledOutage",
    [
        "timestamp",
        "ptid",
        "equipment_name",
        "scheduled_out_datetime",
        "scheduled_in_datetime",
        "bus1",
        "bus2",
        "voltage",
    ],
)
Interval = namedtuple("Interval", ["start", "end"])

equipment_name_pattern = r"^([A-Za-z0-9._]{8})-([A-Za-z0-9._]{8})_(\d{2,3})_(.+)$"

In [5]:
zip_path = actual_files[0]

In [6]:
import io
from datetime import datetime
from zipfile import ZipFile


def parse_actual_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    # Before Feb. 2015 the key had an extra space at the beginning, so try both
    # Before Oct. 2010 exists a trailing space!
    outage_date_time_keys = [
        "Outage Date/Time",
        " Outage Date/Time",
        " Outage Date/Time ",
    ]
    for outage_date_time_key in outage_date_time_keys:
        if outage_date_time_key in row:
            outage_date_time = row[outage_date_time_key]
            break
    return ActualOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        outage_datetime=datetime.strptime(outage_date_time, "%m/%d/%Y %H:%M:%S"),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def parse_scheduled_outage(row):
    equipment_name = row["Equipment Name"]
    m = re.match(equipment_name_pattern, equipment_name)
    if m is None:
        return None
    return ScheduledOutage(
        timestamp=datetime.strptime(row["Timestamp"], "%m/%d/%Y %H:%M:%S"),
        ptid=int(row["PTID"]),
        equipment_name=row["Equipment Name"],
        scheduled_out_datetime=datetime.strptime(
            row["Scheduled Out Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        scheduled_in_datetime=datetime.strptime(
            row["Scheduled In Date/Time"],
            "%m/%d/%Y %H:%M:%S",
        ),
        bus1=m.group(1),
        bus2=m.group(2),
        voltage=int(m.group(3)),
    )


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name, parse_row):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        # Only keep rows with line outages
        data = [parse_row(row) for row in csv_reader]
        data = [row for row in data if row is not None]
    return data


def summarize_data(data):
    """Implement processing method described in this paper titled "Transmission grid outage statistics extracted from a webpage logging outages in northeast america" """
    # group by ptid
    by_ptid = defaultdict(list)
    for outage in data:
        by_ptid[outage.ptid].append(outage)
    # separate ptid by outage date
    by_ptid_and_outage_date = defaultdict(lambda: defaultdict(list))
    for ptid, rows in by_ptid.items():
        for row in rows:
            by_ptid_and_outage_date[ptid][row.outage_datetime].append(row)
        # check partition length be at least two, for beginning and ending
        for partition_key in by_ptid_and_outage_date[ptid].keys():
            partition_list = by_ptid_and_outage_date[ptid][row.outage_datetime]
            if len(partition_list) == 1:
                partition_list.append(partition_list[0])
    # compression by skipping redundancies data
    summary = {}
    for ptid, date_log_dd in by_ptid_and_outage_date.items():
        partitions = {}
        for date, log_list in date_log_dd.items():
            interval = Interval(log_list[0], log_list[-1])
            partitions[date] = interval
        summary[ptid] = partitions
    return summary


# Example using an existing variable in the notebook:
zip_path = actual_files[0]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member, parse_actual_outage)
print(len(data))
summary = summarize_data(data)
print(len(summary))

raw-zip-actual/20060101RTLineOutages_csv.zip
members: ['20060101RTLineOutages.csv', '20060102RTLineOutages.csv', '20060103RTLineOutages.csv', '20060104RTLineOutages.csv', '20060105RTLineOutages.csv', '20060106RTLineOutages.csv', '20060107RTLineOutages.csv', '20060108RTLineOutages.csv', '20060109RTLineOutages.csv', '20060110RTLineOutages.csv', '20060111RTLineOutages.csv', '20060112RTLineOutages.csv', '20060113RTLineOutages.csv', '20060114RTLineOutages.csv', '20060115RTLineOutages.csv', '20060116RTLineOutages.csv', '20060117RTLineOutages.csv', '20060118RTLineOutages.csv', '20060119RTLineOutages.csv', '20060120RTLineOutages.csv', '20060121RTLineOutages.csv', '20060122RTLineOutages.csv', '20060123RTLineOutages.csv', '20060124RTLineOutages.csv', '20060125RTLineOutages.csv', '20060126RTLineOutages.csv', '20060127RTLineOutages.csv', '20060128RTLineOutages.csv', '20060129RTLineOutages.csv', '20060130RTLineOutages.csv', '20060131RTLineOutages.csv']
8352
29


In [7]:
from tqdm import tqdm

actual_outages = defaultdict(dict)
for zip_path in tqdm(actual_files):
    for member in list_csvs(zip_path):
        data = read_csv_from_zip(zip_path, member, parse_actual_outage)
        summary = summarize_data(data)
        for ptid, partitions in summary.items():
            for p_key, p_value in partitions.items():
                if p_key in actual_outages[ptid]:
                    existing_interval = actual_outages[ptid][p_key]
                    new_interval = Interval(
                        start=min(existing_interval.start, p_value.start),
                        end=max(existing_interval.end, p_value.end),
                    )
                    actual_outages[ptid][p_key] = new_interval
                else:
                    actual_outages[ptid][p_key] = p_value

100%|██████████| 230/230 [20:52<00:00,  5.44s/it]


In [15]:
actual_outages.keys()

dict_keys([25053, 25094, 25243, 25299, 25507, 25559, 25560, 25563, 25564, 25565, 25566, 25567, 25569, 25769, 25770, 25875, 25876, 26053, 26058, 26123, 26169, 26187, 26256, 26257, 26264, 26473, 26478, 26626, 325239, 25142, 25252, 25132, 25250, 25562, 25150, 26236, 26235, 26212, 25145, 26134, 26493, 25104, 25246, 25556, 25291, 25048, 25312, 25244, 25313, 25533, 25307, 25554, 25269, 26006, 25134, 25561, 25310, 25550, 25279, 100001219, 25190, 25228, 25573, 25878, 25308, 25829, 25879, 25267, 25494, 25491, 25497, 26497, 25540, 25341, 25830, 25508, 25106, 26198, 26035, 26036, 26159, 26056, 25534, 25681, 25535, 26228, 26048, 26026, 26019, 25210, 25340, 25168, 25311, 325166, 26115, 25141, 26112, 25109, 25220, 25538, 25547, 25087, 25870, 26135, 26163, 26255, 26113, 25079, 25315, 26107, 26239, 25262, 25347, 26050, 26015, 25429, 25152, 25090, 25309, 25155, 25201, 25865, 25682, 25725, 25345, 25859, 26097, 26186, 25264, 25881, 26174, 26203, 25553, 26052, 26016, 25868, 25872, 26041, 26074, 25265, 262

In [13]:
actual_outages[25013]

{datetime.datetime(2006, 2, 23, 11, 28): Interval(start=ActualOutage(timestamp=datetime.datetime(2006, 2, 23, 11, 32, 22), ptid=25013, equipment_name='E.SAYRE_-NWAVERLY_115_956', outage_datetime=datetime.datetime(2006, 2, 23, 11, 28), bus1='E.SAYRE_', bus2='NWAVERLY', voltage=115), end=ActualOutage(timestamp=datetime.datetime(2006, 2, 23, 23, 57, 24), ptid=25013, equipment_name='E.SAYRE_-NWAVERLY_115_956', outage_datetime=datetime.datetime(2006, 2, 23, 11, 28), bus1='E.SAYRE_', bus2='NWAVERLY', voltage=115)),
 datetime.datetime(2006, 6, 28, 15, 54): Interval(start=ActualOutage(timestamp=datetime.datetime(2006, 6, 28, 16, 37, 23), ptid=25013, equipment_name='E.SAYRE_-NWAVERLY_115_956', outage_datetime=datetime.datetime(2006, 6, 28, 15, 54), bus1='E.SAYRE_', bus2='NWAVERLY', voltage=115), end=ActualOutage(timestamp=datetime.datetime(2006, 6, 29, 23, 57, 25), ptid=25013, equipment_name='E.SAYRE_-NWAVERLY_115_956', outage_datetime=datetime.datetime(2006, 6, 28, 15, 54), bus1='E.SAYRE_', bu

TypeError: keys must be str, int, float, bool or None, not datetime.datetime